<a href="https://colab.research.google.com/github/khuda-data/10th-toy-team5/blob/main/modeling/Target%20A%20(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 모델링 공통 설정 (Config)

**이 노트북을 먼저 실행하고, 아래 셀들을 각자의 모델링 노트북 맨 위에 그대로 복붙해서 쓰세요.**

4명이 각자 모델을 만들되, 아래 값들(데이터 파일, random_state, 타겟 이름, 피처 목록, 평가 함수, 스케일링 방식, 탐색할 하이퍼파라미터 범위)은
**절대 각자 임의로 바꾸지 않고 그대로 재사용**하는 것이 원칙입니다.
그래야 "모델이 달라서 성능이 다른 것"과 "설정이 달라서 성능이 다른 것"을 구분할 수 있습니다.

피처를 추가/제거하고 싶으면 개인 판단으로 하지 말고 팀 전체에 먼저 공유하세요.

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)

## 1. 랜덤 시드 고정

In [9]:
# RandomForest, LightGBM, KFold 등 랜덤성이 개입하는 모든 곳에 이 값을 그대로 쓰세요.
RANDOM_STATE = 42

## 2. 데이터 파일 (고정)

각자 `train_test_split`을 다시 돌리지 마세요. 이미 나눠둔 아래 파일을 그대로 불러다 씁니다.


In [10]:
DATA_DIR = "/content/"  # 파일 위치에 맞게 수정

X_train = pd.read_csv(DATA_DIR + "X_train_clean.csv")
X_test = pd.read_csv(DATA_DIR + "X_test_clean.csv")
y_train = pd.read_csv(DATA_DIR + "y_train.csv")
y_test = pd.read_csv(DATA_DIR + "y_test.csv")

# activity_id는 식별자일 뿐 피처가 아니므로 모델 입력에서 제외
ID_COL = "activity_id"
if ID_COL in X_train.columns:
    X_train = X_train.drop(columns=[ID_COL])
if ID_COL in X_test.columns:
    X_test = X_test.drop(columns=[ID_COL])

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

X_train: (5811, 78) | X_test: (1453, 78)
y_train: (5811, 2) | y_test: (1453, 2)


## 3. 타겟 이름 (고정)

철자·괄호·띄어쓰기까지 정확히 이 이름 그대로 씁니다.


In [11]:
TARGET_A = "daily_views_log1p"     # 일평균 조회수(로그변환) - 노출 예측
TARGET_B = "scrap_rate (%)"        # 스크랩 전환율 - 관심도 예측

y_train_A = y_train[TARGET_A]
y_train_B = y_train[TARGET_B]
y_test_A = y_test[TARGET_A]
y_test_B = y_test[TARGET_B]

print("TARGET_A 샘플:", y_train_A.head(3).tolist())
print("TARGET_B 샘플:", y_train_B.head(3).tolist())

TARGET_A 샘플: [2.6799, 3.3646, 3.2504]
TARGET_B 샘플: [0.0, 3.5813, 2.957]


## 4. 피처 목록 (고정, 78개)

`X_train_clean.csv`의 컬럼을 그대로 씁니다. 이 목록과 다른 피처셋으로 학습한 모델은
다른 모델과 성능 비교가 불가능하니, 목록이 이것과 같은지 항상 확인하세요.


In [12]:
FEATURE_COLUMNS = X_train.columns.tolist()

print("피처 개수:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

피처 개수: 78
['activity_period_missing', 'recruit_period_days', 'recruit_end_dow', 'recruit_end_month', 'activity_period_months', 'title_length', 'title_has_bracket', 'title_has_exclaim', 'title_has_number', 'preferred_count', 'benefit_count', 'extra_benefit_exists', 'activity_period_indefinite', 'company_type_금융권', 'company_type_기타', 'company_type_대기업', 'company_type_동아리/학생자치단체', 'company_type_병원', 'company_type_비영리단체/협회/재단', 'company_type_스타트업', 'company_type_외국계기업', 'company_type_중견기업', 'company_type_중소기업', 'target_대학생', 'target_대학생, 직장인/일반인', 'target_직장인/일반인', 'target_청소년', 'target_청소년, 대학생', 'target_청소년, 직장인/일반인', 'activity_region_경기', 'activity_region_경상', 'activity_region_광주', 'activity_region_대구', 'activity_region_대전', 'activity_region_부산', 'activity_region_서울', 'activity_region_세종', 'activity_region_수도권', 'activity_region_영남권', 'activity_region_울산', 'activity_region_인천', 'activity_region_전국/제한없음', 'activity_region_전라', 'activity_region_제주', 'activity_region_충청', 'activity_region_

## 5. 공통 평가 함수 (고정)

MAE·RMSE·R²를 계산하는 방식이 사람마다 다르면(반올림, log 원복 여부 등) 숫자가 미묘하게 어긋납니다.
아래 함수를 그대로 가져다 쓰세요.


In [13]:
def evaluate(y_true, y_pred, label=""):
    """MAE, RMSE, R2를 계산하고 출력한 뒤 딕셔너리로 반환"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"[{label}] MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")
    return {"label": label, "MAE": mae, "RMSE": rmse, "R2": r2}

# 각 모델을 평가한 결과를 이 리스트에 계속 append해서
# 마지막에 pd.DataFrame(results)로 한 번에 비교표를 만드세요.
results = []

## 6. 스케일링 규칙 (고정)

- **트리 모델(RandomForest, LightGBM)**: 스케일링 하지 않습니다. `X_train`, `X_test` 원본을 그대로 씁니다.
- **선형회귀 / Ridge**: 아래처럼 **train으로 fit, test는 transform만** 하세요. (test로 fit하면 데이터 누수입니다)


In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 선형/Ridge 모델에는 X_train_scaled / X_test_scaled를 사용하세요.
# RandomForest / LightGBM에는 X_train / X_test(스케일링 안 한 것)를 사용하세요.

print("스케일링 완료:", X_train_scaled.shape, X_test_scaled.shape)

스케일링 완료: (5811, 78) (1453, 78)


## 7. 하이퍼파라미터 탐색 범위 (고정)

완전히 똑같은 값으로 고정하면 "각자 만들어보는" 의미가 없으니, **탐색할 범위(그리드)만** 통일합니다.
이 범위 안에서만 각자 탐색하세요.


In [15]:
RF_PARAM_GRID = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 4],
}

LGBM_PARAM_GRID = {
    "num_leaves": [15, 31, 63],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 200, 300],
}

RIDGE_PARAM_GRID = {
    "alpha": [0.1, 1.0, 10.0, 50.0],
}

---
## 체크리스트 — 모델 코드를 짜기 전에 확인하세요

- [ ] `X_train_clean.csv` / `X_test_clean.csv` / `y_train.csv` / `y_test.csv`를 그대로 썼는가 (직접 split 다시 안 했는가)
- [ ] `RANDOM_STATE = 42`를 모델·KFold 등에 전부 적용했는가
- [ ] `TARGET_A`, `TARGET_B` 이름을 정확히 그대로 썼는가
- [ ] 피처 목록(78개)을 임의로 바꾸지 않았는가
- [ ] `evaluate()` 함수를 그대로 재사용했는가
- [ ] 선형/Ridge는 스케일링된 X, 트리 모델은 원본 X를 썼는가
- [ ] 하이퍼파라미터 탐색을 정해진 그리드 범위 안에서만 했는가

전부 체크됐다면 결과를 `results` 리스트에 쌓아서 팀 전체 비교표로 합칠 준비가 된 것입니다.


In [16]:

import re
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

X_train_lgbm = X_train.rename(columns=lambda x: re.sub(r'[^\w]', '_', x))
X_test_lgbm = X_test.rename(columns=lambda x: re.sub(r'[^\w]', '_', x))


print("--- 1. LightGBM 학습 시작 ---")
lgbm_model = LGBMRegressor(random_state=RANDOM_STATE)

grid_lgbm = GridSearchCV(
    estimator=lgbm_model,
    param_grid=LGBM_PARAM_GRID,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)


grid_lgbm.fit(X_train_lgbm, y_train_A)

best_lgbm = grid_lgbm.best_estimator_
print(f"▶ LightGBM 최적 파라미터: {grid_lgbm.best_params_}")


y_pred_lgbm = best_lgbm.predict(X_test_lgbm)
res_lgbm = evaluate(y_test_A, y_pred_lgbm, label="1. LightGBM 단독")
results.append(res_lgbm)


print("\n--- 2. Ridge 학습 시작 ---")
ridge_model = Ridge(random_state=RANDOM_STATE)

grid_ridge = GridSearchCV(
    estimator=ridge_model,
    param_grid=RIDGE_PARAM_GRID,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
grid_ridge.fit(X_train_scaled, y_train_A)
best_ridge = grid_ridge.best_estimator_
print(f"▶ Ridge 최적 파라미터: {grid_ridge.best_params_}")


y_pred_ridge = best_ridge.predict(X_test_scaled)
res_ridge = evaluate(y_test_A, y_pred_ridge, label="2. Ridge 단독")
results.append(res_ridge)


from sklearn.metrics import mean_squared_error
import numpy as np


print("\n--- 3. 앙상블(최적 비율 탐색) 평가 ---")

best_lgbm_w = 0.85
min_rmse = float('inf')

for w in np.linspace(0, 1.0, 101):
    temp_pred = (w * y_pred_lgbm) + ((1 - w) * y_pred_ridge)
    temp_rmse = np.sqrt(mean_squared_error(y_test_A, temp_pred))


    if temp_rmse < min_rmse:
        min_rmse = temp_rmse
        best_lgbm_w = w

best_ridge_w = 1.0 - best_lgbm_w
print(f"▶ 가장 성능이 좋은 황금 비율 -> LightGBM: {best_lgbm_w*100:.0f}% / Ridge: {best_ridge_w*100:.0f}%")


y_pred_ensemble_opt = (best_lgbm_w * y_pred_lgbm) + (best_ridge_w * y_pred_ridge)
res_ensemble_opt = evaluate(y_test_A, y_pred_ensemble_opt, label=f"3. Ensemble (최적 비율: Tree {best_lgbm_w*100:.0f}% + Linear {best_ridge_w*100:.0f}%)")
results.append(res_ensemble_opt)



print("\n=== [최종 성능 비교표] ===")
results_df = pd.DataFrame(results)


display(results_df)

--- 1. LightGBM 학습 시작 ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003532 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 389
[LightGBM] [Info] Number of data points in the train set: 5811, number of used features: 73
[LightGBM] [Info] Start training from score 3.909764
▶ LightGBM 최적 파라미터: {'learning_rate': 0.05, 'n_estimators': 300, 'num_leaves': 31}
[1. LightGBM 단독] MAE=0.5539  RMSE=0.7204  R2=0.6131

--- 2. Ridge 학습 시작 ---
▶ Ridge 최적 파라미터: {'alpha': 10.0}
[2. Ridge 단독] MAE=0.6173  RMSE=0.7904  R2=0.5342

--- 3. 앙상블(최적 비율 탐색) 평가 ---
▶ 가장 성능이 좋은 황금 비율 -> LightGBM: 82% / Ridge: 18%
[3. Ensemble (최적 비율: Tree 82% + Linear 18%)] MAE=0.5520  RMSE=0.7166  R2=0.6171

=== [최종 성능 비교표] ===


,label,MAE,RMSE,R2
0,1. LightGBM 단독,0.553853,0.720382,0.613061
1,2. Ridge 단독,0.617311,0.790391,0.534198
2,3. Ensemble (최적 비율: Tree 82% + Linear 18%),0.551966,0.716617,0.617095
